# 03d — Qwen3-ASR-0.6B: LoRA recipe revised after the 03c results

A copy of `03c` (encoder-focused LoRA + clean/noisy consistency loss) whose training and model
selection follow from what the 03c run on FLEURS showed. Data tables, noise schedule, decoding rule
(unless the optional loop guard is on), scoring and output names follow notebooks 03 – 03c. Before
running: *Settings → Accelerator: GPU T4*, *Internet: on*, and *Add Input* → the shared dataset
**`asr-noisy-vi`**. The GitHub repo holds only the code; the audio and manifests come from the dataset.

**What 03c showed** (FLEURS test, 850 utterances per condition, WER %)

| | clean | traffic 20 / 10 / 5 / 0 dB | babble 20 / 10 / 5 / 0 dB |
|---|---|---|---|
| zero-shot | 8.33 | 8.81 / 10.09 / 12.87 / 19.73 | 8.66 / 13.28 / 30.65 / 109.59 |
| 03c fine-tuned | 8.99 | 9.41 / 11.10 / 14.36 / 23.29 | 9.51 / 12.88 / 22.50 / 95.18 |

- Better only where the zero-shot model loops (babble 5 and 0 dB); slightly but significantly worse
  on clean speech, every traffic level and babble 20 dB.
- Over-fitting: the training loss fell from 0.29 to 0.08 while the dev WER stayed near 20–21. The
  best epoch (2) came at 89% of the peak learning rate of a 10-epoch schedule.
- Model selection used 100 dev files, where a single looping transcript (word error rate above
  1000%) moves the dev WER by points. The short sweep (half the data, one epoch, learning rate
  decayed to 0) scored better on dev than the full run of its winner.
- The consistency loss helped in the sweep: dev WER 18.7 with it, 20.3 and 31.1 without.

**What changed from 03c**

| | 03c | This notebook | Why |
|---|---|---|---|
| Dev set for model selection | 100 files (`DEV_LIMIT = 100`) | the whole dev set (`DEV_LIMIT = None`) | fewer chance swings from single looping transcripts |
| Epochs | up to 10, early stopping | `EPOCHS = 3`: the learning rate decays to 0 within them | 03c over-fitted, and its best checkpoint never saw a low learning rate |
| Recipe choice (7) | short sweep, then a full run of the winner | every candidate trains the full schedule; the one with the lowest dev WER is tested | the short sweep favoured the learning rate that over-fitted in the full run |
| Candidates | LoRA layers × consistency | all with the consistency loss: lr 1e-4, lr 5e-5, encoder-only LoRA, and the 03c recipe (lr 2e-4) as a control | forget less of what the base model already does well |
| Output names | `..._ft-noisy` for any recipe | `..._ft-noisy-<recipe tag>` | each recipe is its own row in the results table |
| Memory | batch 8 × 2: out of memory on the T4 | batch 4 × 4, gradient checkpointing, `expandable_segments` (03c: peak 5.5 GB) | |
| Restart check (6.1) | — | stops when the GPU still holds memory from an earlier failed cell | re-running cells after an out-of-memory error loaded a second model |
| Data folder (2.1) | `DATASET.json` needed | falls back to the folder holding `manifests/clean.csv` | the shared copy has no `DATASET.json` |
| Loop guard (3.1) | — | optional `CAP_TOKENS_BY_DURATION`: about twice the usual number of tokens for the audio's duration at most; outputs get a `-cap` suffix | most babble 0 dB errors are transcripts that loop until `MAX_NEW_TOKENS` |

Unchanged: training data (`DATASETS`), `train_schedule.csv`, the test and dev tables, scoring, 16
utterances per update. The loop guard changes the decoding rule: if it is on, decode every model you
compare (PhoWhisper included) the same way. Every choice is made on dev, never on test.

**Time on a T4** (from the 03c run): zero-shot test about 1 h; one candidate about 45 min (3 epochs of
about 12 min, plus the dev passes); fine-tuned test about 1 h. With 4 candidates: about 5 h. Remove
candidates from `CANDIDATES` to go faster; a finished candidate is not trained again when 7.1 is
re-run in the same session.

| Step | What | Output in `/kaggle/working` |
|---|---|---|
| 0 | Configuration | |
| 1 | Clone the repo, install it, make sure `transformers` knows Qwen3-ASR | |
| 2 | Find the dataset, load the test / dev tables (9 conditions: clean, traffic and babble at 20/10/5/0 dB) | |
| 3 | Load Qwen3-ASR-0.6B, `transcribe(model, processor, audios)` (optional loop guard), shared helpers | |
| 4 | Sanity check on 10 files, then the full zero-shot test | `predictions_qwen3-asr-0.6b_zero-shot[-cap].csv` |
| 5 | Evaluate: WER / CER per noise × SNR, bootstrap intervals | |
| 6 | Fine-tuning setup: training set, LoRA with per-part ranks, collate (with the clean features for the consistency loss), restart check, memory and overfit checks | |
| 7 | 7.0 training function · 7.1 train every candidate · 7.2 choose the lowest dev WER | `lora_*/`, `history_*.csv`, `candidate_*.json`, `candidates_*.csv`, `run_*.json` |
| 8 | Test the chosen adapter, compare with zero-shot | `predictions_qwen3-asr-0.6b_ft-noisy-<tag>[-cap].csv` |

After step 2 the code only sees `(rows, audios)`: 16 kHz float32 arrays and normalized
reference text. Steps 6–8 can run in a later session: run steps 0–3, then jump to 6.

**What is specific to Qwen3-ASR** (as in notebook 03):

- **Checkpoint**: `Qwen/Qwen3-ASR-0.6B-hf`, loaded with the classes built into `transformers`
  (5.13.0 or newer). The `qwen-asr` package is not used: it pins `transformers==4.57.6`.
- **Prompt**: the model is a speech encoder followed by a Qwen3 language model. It reads a chat
  prompt that holds the audio and ends with `language Vietnamese<asr_text>`, then writes the
  transcript and the end-of-turn token `<|im_end|>`.
- **Training target**: the same prompt as at test time; the loss covers only the transcript
  and `<|im_end|>`.
- **Precision**: weights stay in float32. Forward passes use float16 on the T4 (it has no
  bfloat16) and bfloat16 on newer GPUs.

## 0. Configuration

In [1]:
# 0.1 Configuration: code commit, model, decoding, fine-tuning
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation; set before torch starts CUDA

# Code (only core / text / scoring helpers are used)
REPO_URL = "https://github.com/anhvinhphan659/asr-noisy-vi"
COMMIT = "83fad8741d1cb742f26250a1f23dd6880c764b60"
CODE_DIR = Path("/tmp/asr-noisy-vi")

# Model and decoding (fixed for every experiment)
MODEL = "qwen3-asr-0.6b"
MODEL_ID = "Qwen/Qwen3-ASR-0.6B-hf"
MIN_TRANSFORMERS = "5.13.0"  # first release with the Qwen3-ASR classes
LANGUAGE = "Vietnamese"      # forced: the prompt ends with "language Vietnamese<asr_text>"
NUM_BEAMS = 1                # greedy
MAX_NEW_TOKENS = 400
EVAL_BATCH = 16
MIXED_PRECISION = True       # False: float32 everywhere (slower, more memory); use it if float16 gives NaN

# Optional loop guard (03c: most babble 0 dB errors are transcripts that loop until MAX_NEW_TOKENS).
# On: a batch writes at most CAP_FACTOR x (99th percentile of tokens per second in the training
# transcripts) x its longest duration + CAP_MARGIN tokens. This changes the decoding rule, so the
# zero-shot and fine-tuned outputs get a "-cap" suffix; compare them only with models decoded the same way.
CAP_TOKENS_BY_DURATION = False
CAP_FACTOR, CAP_MARGIN = 2.0, 10
DECODE_TAG = "-cap" if CAP_TOKENS_BY_DURATION else ""
ZS_VARIANT = "zero-shot" + DECODE_TAG

# Fine-tuning: fixed, as in notebook 03 (fair comparison with PhoWhisper)
VARIANT = "ft-noisy"         # "ft-noisy" (online noise from train_schedule.csv) or "ft-clean"
TRAIN_BATCH = 4              # utterances per forward pass (8 ran out of memory on the T4)
GRAD_ACCUM = 4               # forward passes per update: 4 x 4 = 16 utterances, as for PhoWhisper
WARMUP_FRACTION = 0.1
LORA_DROPOUT = 0.05
MAX_LABEL_TOKENS = 448
SEED = 42

# Fine-tuning: schedule and model selection (03c: 10-epoch schedule over-fitted, 100 dev files were too few)
EPOCHS = 3                   # the learning rate decays to 0 within these epochs (<= 10, the length of train_schedule.csv)
PATIENCE = 2                 # stop after this many epochs without a lower dev WER
DEV_LIMIT = None             # dev WER on the whole dev set; a number: that many utterances per dataset
TRAIN_FRACTION = 1.0         # part of the training set every candidate trains on; < 1 for a quick run

# Fine-tuning: speed and memory
GROUP_BY_LENGTH = True       # batches of utterances of similar duration (less padding); False: random, as in 03
GRAD_CHECKPOINT = True       # less GPU memory, about 30% slower (03c: peak 5.5 GB with TRAIN_BATCH = 4)
NUM_WORKERS = min(4, os.cpu_count() or 1)   # the online noise mixing runs in these CPU workers

# Fine-tuning: recipe = LoRA layers, rank (one for every layer, or one for the speech encoder and
# one for the language model), learning rate, SpecAugment, weight of the clean/noisy consistency loss
LORA_QV = ["q_proj", "v_proj"]                       # notebook 03
LORA_ALL = ["q_proj", "k_proj", "v_proj",            # attention, speech encoder and language model
            "out_proj", "o_proj",                    # attention output: encoder / language model
            "fc1", "fc2",                            # MLP of the speech encoder
            "gate_proj", "up_proj", "down_proj"]     # MLP of the language model
# The speech encoder's layers, for rank_pattern / alpha_pattern. peft matches the pattern against
# the end of the module name inside the base model ("model.audio_tower.layers.3.self_attn.q_proj"),
# so the trailing ".*" is needed. Here rank_pattern is for the encoder only: add_lora checks that
# the encoder gets the pattern's rank and every other adapted layer gets r.
ENCODER_PATTERN = "audio_tower.*"
# Only the speech encoder's projections: a string is a regular expression that peft matches against
# the whole module name inside the base model. The language model keeps its weights.
ENCODER_ONLY_TARGETS = r".*audio_tower\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|out_proj)|fc1|fc2)"
ALL_R16 = dict(targets=LORA_ALL, r=16, alpha=32, rank_pattern={}, alpha_pattern={})   # the 03b recipe
ENCODER_FOCUS = dict(targets=LORA_ALL, r=8, alpha=16,                                 # language model: r = 8
                     rank_pattern={ENCODER_PATTERN: 32}, alpha_pattern={ENCODER_PATTERN: 64})
ENCODER_ONLY = dict(targets=ENCODER_ONLY_TARGETS, r=32, alpha=64, rank_pattern={}, alpha_pattern={})

# Candidates (7.1): each trains EPOCHS on TRAIN_FRACTION of the training set; 7.2 tests the one with
# the lowest dev WER; `tag` names its outputs. 03c: the consistency loss helped and lr 2e-4 over-fitted,
# so every candidate keeps the consistency loss and the first three learn more gently.
CONSISTENCY = 1.0            # weight of the clean/noisy consistency loss; 0 turns it off
CANDIDATES = [
    dict(name="encoder r32 / LM r8 + consistency, lr 1e-4", tag="enc32-lm8-cons-lr1e-4",
         **ENCODER_FOCUS, lr=1e-4, augment=False, consistency=CONSISTENCY),
    dict(name="encoder r32 / LM r8 + consistency, lr 5e-5", tag="enc32-lm8-cons-lr5e-5",
         **ENCODER_FOCUS, lr=5e-5, augment=False, consistency=CONSISTENCY),
    dict(name="encoder-only r32 + consistency, lr 1e-4", tag="enc32-only-cons-lr1e-4",
         **ENCODER_ONLY, lr=1e-4, augment=False, consistency=CONSISTENCY),
    dict(name="encoder r32 / LM r8 + consistency, lr 2e-4 (03c recipe)", tag="enc32-lm8-cons-lr2e-4",
         **ENCODER_FOCUS, lr=2e-4, augment=False, consistency=CONSISTENCY),
]
RUN = CANDIDATES[0]          # recipe of the checks in 6.1-6.3 (LoRA in both parts + consistency: the most memory)

# SpecAugment (recipes with augment=True), on the 128-bin log-mel features
SPEC_FREQ_MASKS, SPEC_FREQ_WIDTH = 2, 27       # 2 bands of up to 27 mel bins
SPEC_TIME_MASKS, SPEC_TIME_FRACTION = 2, 0.05  # 2 spans of up to 5% of the utterance each

SR = 16000
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Code: clone and install

If the repo is private, add a Kaggle secret `GITHUB_TOKEN` (*Add-ons → Secrets*).

`pip` upgrades `transformers` only when the installed one is older than `MIN_TRANSFORMERS`.
If cell 1.3 then reports an old version, `transformers` was imported before the upgrade:
restart the kernel and run again from 0.1.

In [2]:
# 1.1 Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# 1.2 Clone the repo at COMMIT and install it
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _token = None
_url = REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL

!rm -rf {CODE_DIR}
!git clone -q {_url} {CODE_DIR}
!git -C {CODE_DIR} checkout -q {COMMIT}
!git -C {CODE_DIR} log -1 --format="%h %ad %s" --date=short
!pip install -q -e "{CODE_DIR}[notebooks]" peft "transformers>={MIN_TRANSFORMERS}"
!pip uninstall -y -q torchao      # Kaggle's torchao is too old for recent peft; not used here

83fad87 2026-11-01 Merge pull request #2 from anhvinhphan659/feature/dataprep-evaluation
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for asr-noisy-vi (pyproject.toml) ... done


In [4]:
# 1.3 Imports and device
import os
import sys
sys.path.insert(0, str(CODE_DIR))      # importable in this kernel without a restart
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # the tokenizer is used in DataLoader workers

import gc
import math
import re

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import Audio, display
from packaging.version import Version
from tqdm.auto import tqdm

from core import load_audio
from dataprep import (collate_raw, describe, find_data_root, iter_eval_batches, load_eval_set,
                      load_train_set, require_data)
from evaluation import PREDICTION_COLUMNS, bootstrap_ci, compare, score, summarize

assert Version(transformers.__version__) >= Version(MIN_TRANSFORMERS), (
    f"transformers {transformers.__version__} is loaded but {MIN_TRANSFORMERS}+ is needed: "
    "restart the kernel and run again from 0.1")

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = MIXED_PRECISION and DEVICE == "cuda"
# bfloat16 needs compute capability 8+ (Ampere or newer); the T4 is 7.5 and uses float16
AMP_DTYPE = torch.bfloat16 if AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
torch.manual_seed(SEED)
print(DEVICE, torch.__version__, "| transformers", transformers.__version__,
      "| autocast", AMP_DTYPE if AMP else "off")

cuda 2.11.0+cu130 | transformers 5.20.2 | autocast torch.float16


## 2. Data

Cell 1 finds the shared dataset with `find_data_root()`: the first folder under `/kaggle/input`
that holds `DATASET.json`, wherever Kaggle mounts it. A copy without `DATASET.json` is found by its
`manifests/clean.csv` instead (the completeness check is then skipped). If nothing is attached it
stops with instructions and the list of folders it looked in. Cell 2 picks what to test (uncomment a
line) and builds `test_set` / `dev_set`: one row per file, with `path` (noisy or clean audio)
and `text` (normalized reference). It also builds `dev_fast`, the dev rows of at most
`DEV_LIMIT` utterances per dataset (all their conditions), for dev WER during training. The
next steps only use these tables and `iter_batches`.

In [5]:
# 2.1 Data folder: the folder under /kaggle/input that holds DATASET.json, or else manifests/clean.csv
!ls /kaggle/input /kaggle/input/datasets 2>/dev/null


def find_manifest_root(roots, max_depth=6):
    # The first folder holding manifests/clean.csv, for a copy of the dataset without DATASET.json
    for root in roots:
        for depth in range(max_depth + 1):
            hits = sorted(Path(root).glob("/".join(["*"] * depth + ["manifests", "clean.csv"])))
            if hits:
                return hits[0].parents[1]
    return None


try:
    DATA_ROOT = find_data_root()   # also looks at $ASR_DATA_ROOT and /kaggle/working/asr-data
except FileNotFoundError:
    DATA_ROOT = find_manifest_root([p for p in (os.environ.get("ASR_DATA_ROOT"), "/kaggle/input") if p])
    if DATA_ROOT is None:
        raise
# DATA_ROOT = Path("/kaggle/input/datasets/tuchan0731/asr-noisy-vi/asr-data")   # or set it by hand
print("DATA_ROOT:", DATA_ROOT)

M = DATA_ROOT / "manifests"
!ls {DATA_ROOT} {M}
if (DATA_ROOT / "DATASET.json").exists():
    print(describe(require_data(DATA_ROOT)))  # full check when DATASET.json is there
else:
    print("no DATASET.json: completeness check skipped")
clean = pd.read_csv(M / "clean.csv")

/kaggle/input:
datasets

/kaggle/input/datasets:
tuchan0731
DATA_ROOT: /kaggle/input/datasets/tuchan0731/asr-noisy-vi/asr-data
/kaggle/input/datasets/tuchan0731/asr-noisy-vi/asr-data:
clean  manifests  noise  noisy

/kaggle/input/datasets/tuchan0731/asr-noisy-vi/asr-data/manifests:
clean.csv	   clean_vivos.csv     dropped_vietmed.csv  noise_voices.csv
clean_fleurs.csv   dev_plan.csv        dropped_vivos.csv    test_plan.csv
clean_vietmed.csv  dropped_fleurs.csv  noise_traffic.csv    train_schedule.csv
no DATASET.json: completeness check skipped


In [6]:
# 2.2 Pick what to test: uncomment one line in each group
DATASETS = ["fleurs"]
# DATASETS = ["vivos"]
# DATASETS = ["vietmed"]
# DATASETS = ["fleurs", "vivos", "vietmed"]

CONDITIONS = None                                            # all 9 conditions
# CONDITIONS = ["clean"]
# CONDITIONS = ["clean", "traffic_5dB", "babble_5dB"]
# CONDITIONS = ["clean", "traffic_20dB", "traffic_10dB", "traffic_5dB", "traffic_0dB"]
# CONDITIONS = ["clean", "babble_20dB", "babble_10dB", "babble_5dB", "babble_0dB"]

LIMIT = None                                                 # all utterances
# LIMIT = 20                                                 # quick run: 20 utterances per dataset

test_set = load_eval_set(M / "test_plan.csv", clean, datasets=DATASETS, conditions=CONDITIONS, limit=LIMIT)
dev_set = load_eval_set(M / "dev_plan.csv", clean, datasets=DATASETS, limit=LIMIT)
# Dev rows for dev WER during training: at most DEV_LIMIT utterances per dataset, all their conditions
_limits = [v for v in (LIMIT, DEV_LIMIT) if v is not None]
dev_fast = load_eval_set(M / "dev_plan.csv", clean, datasets=DATASETS, limit=min(_limits) if _limits else None)
iter_batches = lambda table, batch_size: iter_eval_batches(table, DATA_ROOT, batch_size=batch_size)

print(f"test: {len(test_set)} files, {test_set['utt_id'].nunique()} utterances, "
      f"{test_set['duration_s'].sum() / 3600:.1f} h | dev: {len(dev_set)} files, "
      f"{len(dev_fast)} during training")
display(test_set.groupby(["dataset", "condition"]).size().unstack(0))
row = test_set.iloc[-1]
print(row["condition"], "|", row["text"])
Audio(str(DATA_ROOT / row["path"]))

test: 7650 files, 850 utterances, 26.4 h | dev: 359 files, 359 during training


dataset,fleurs
condition,
babble_0dB,850
babble_10dB,850
babble_20dB,850
babble_5dB,850
clean,850
traffic_0dB,850
traffic_10dB,850
traffic_20dB,850
traffic_5dB,850


babble_0dB | những nhà thờ truyền thống hơn thường tổ chức lễ vọng phục sinh vào đêm thứ bảy trong kỳ cuối tuần của lễ phục sinh và những người tụ họp thường cùng nhau chúc mừng vào lúc nửa đêm để mừng chúa phục sinh


## 3. Model and `transcribe`

Decoding is fixed for every experiment: Vietnamese forced in the prompt, greedy
(`num_beams=1`, no sampling), at most `MAX_NEW_TOKENS` tokens. `generate` returns the prompt
followed by the answer, so the prompt is cut off before decoding. The output is returned as
the model writes it (Qwen3-ASR writes capitals and punctuation, and may write numbers as
digits); scoring normalizes both sides. The processor's decoder collapses a character or short pattern repeated more
than 20 times, as the original Qwen3-ASR code does.

**Loop guard** (`CAP_TOKENS_BY_DURATION`, off by default). Under strong babble the model can repeat
a phrase until `MAX_NEW_TOKENS` (03c: transcripts with word error rates above 1000%). With the guard
on, a batch writes at most `CAP_FACTOR` × the 99th percentile of tokens per second in the training
transcripts × its longest duration + `CAP_MARGIN` tokens; batches hold utterances of similar
length. A normal transcript stays well inside the limit, and a loop is cut at about twice the
expected length. The outputs get a `-cap` suffix, because this is a different decoding rule.

In [7]:
# 3.1 Load Qwen3-ASR-0.6B and define transcribe(); quick test on one file
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration

DECODE = dict(num_beams=NUM_BEAMS, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Loop guard (CAP_TOKENS_BY_DURATION): tokens per second of speech in the training transcripts
TOKENS_PER_SECOND = None
if CAP_TOKENS_BY_DURATION:
    _train = clean[(clean["split"] == "train") & clean["dataset"].isin(DATASETS)]
    _tokens = np.array([len(ids) for ids in processor.tokenizer(_train["text"].tolist(),
                                                               add_special_tokens=False).input_ids])
    TOKENS_PER_SECOND = float(np.percentile(_tokens / _train["duration_s"].to_numpy(), 99))
    print(f"loop guard: at most {CAP_FACTOR} x {TOKENS_PER_SECOND:.1f} tokens/s x duration + {CAP_MARGIN} tokens")


def max_new_tokens(audios):
    if TOKENS_PER_SECOND is None:
        return MAX_NEW_TOKENS
    longest_s = max(len(a) for a in audios) / SR
    return min(MAX_NEW_TOKENS, math.ceil(CAP_FACTOR * TOKENS_PER_SECOND * longest_s) + CAP_MARGIN)


def load_base_model():
    # The checkpoint is stored in bfloat16; float32 weights + autocast work on every GPU.
    model = Qwen3ASRForConditionalGeneration.from_pretrained(MODEL_ID, dtype=torch.float32)
    return model.to(DEVICE).eval()


@torch.no_grad()
def transcribe(model, processor, audios):
    # list of 16 kHz float32 arrays -> list of raw transcripts
    audios = list(audios)
    inputs = processor.apply_transcription_request(audio=audios, language=LANGUAGE).to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP):
        ids = model.generate(**inputs, **{**DECODE, "max_new_tokens": max_new_tokens(audios)})
    answers = ids[:, inputs["input_ids"].shape[1]:]          # drop the prompt
    return [text.strip() for text in processor.decode(answers, return_format="transcription_only")]


model = load_base_model()
print(transcribe(model, processor, [load_audio(DATA_ROOT / test_set["path"].iloc[0])]))

processor_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

['Văn hóa và bộ lạc cổ xưa đã bắt đầu giữ những con vật này để dễ lấy sữa, tóc, thịt và da.']


### Shared helpers: run a model on a table, evaluate predictions

`evaluate` reuses `evaluation.score` (same `normalize_text` on reference and hypothesis) and
`summarize` (corpus-level WER / CER in percent, split into substitutions, deletions,
insertions).

In [8]:
# 3.2 Shared helpers: run a model on a table, check outputs, evaluate, tables
VI_DIACRITICS = re.compile("[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]",
                           re.IGNORECASE)
# Latin letters with Vietnamese diacritics, digits, common punctuation
ALLOWED = re.compile(r"[\s0-9A-Za-zÀ-ɏ̀-ͯḀ-ỿ.,!?;:'\"()%&/\-–…]")
SNR_ORDER = ["clean", "20 dB", "10 dB", "5 dB", "0 dB"]


def run_predictions(model, table, variant, batch_size=EVAL_BATCH, desc=None):
    rows, hyps = [], []
    for batch_rows, audios in tqdm(iter_batches(table, batch_size),
                                   total=math.ceil(len(table) / batch_size), desc=desc or variant):
        rows.append(batch_rows)
        hyps += transcribe(model, processor, audios)
    out = pd.concat(rows)[["utt_id", "condition", "dataset", "noise_type", "snr_db", "text"]]
    return out.assign(model=MODEL, variant=variant, hyp=hyps).reset_index(drop=True)


def predictions_path(variant):
    return OUT / f"predictions_{MODEL}_{variant}.csv"


def run_or_load(model, table, variant, path=None, force=False):
    # Reuse the saved CSV when it covers exactly this table (unless force); otherwise run and save.
    # `model` may also be a function that returns a model: it is called only when the CSV is not reused.
    path = path or predictions_path(variant)
    if path.exists() and not force:
        saved = pd.read_csv(path, dtype={"hyp": str}, keep_default_na=False)
        if set(zip(saved["utt_id"], saved["condition"])) == set(zip(table["utt_id"], table["condition"])):
            print(f"loaded {path}")
            return saved
    if not isinstance(model, torch.nn.Module):
        model = model()
    pred = run_predictions(model, table, variant)
    pred.to_csv(path, index=False)
    print(f"wrote {path} ({len(pred)} rows)")
    return pred


def check_outputs(pred):
    report = pred[["utt_id", "condition", "text", "hyp"]].copy()
    report["diacritics"] = report["hyp"].str.contains(VI_DIACRITICS)
    report["odd_chars"] = report["hyp"].map(lambda t: "".join(sorted({c for c in t if not ALLOWED.match(c)})))
    report["special_token"] = report["hyp"].str.contains("<|", regex=False)
    report["empty"] = report["hyp"].str.strip().eq("")
    return report


def evaluate(pred, table, by=("model", "variant", "dataset", "noise_type", "snr_db")):
    plan = table[["utt_id", "condition", "dataset", "noise_type", "snr_db"]]
    refs = table[["utt_id", "text"]].drop_duplicates("utt_id")
    scored = score(pred[PREDICTION_COLUMNS], plan, refs)
    return scored, summarize(scored, by=list(by))


def wer_grid(summary, metric="wer"):
    # One row per system × noise type, one column per SNR.
    s = summary.assign(snr=summary["snr_db"].map(lambda v: "clean" if pd.isna(v) else f"{v:g} dB"))
    grid = s.pivot_table(index=["model", "variant", "dataset", "noise_type"], columns="snr",
                         values=metric, aggfunc="first")
    return grid[[c for c in SNR_ORDER if c in grid.columns]]


def worst(scored, n=5):
    errors = scored[["w_sub", "w_del", "w_ins"]].sum(axis=1) / scored["w_n_ref"].clip(lower=1)
    return scored.assign(utt_wer=(100 * errors).round(1)).nlargest(n, "utt_wer")[
        ["utt_id", "condition", "utt_wer", "ref", "hyp_norm"]]

## 4. Zero-shot test

Sanity check on 10 clean files (read every transcript), then the full test table.

In [9]:
# 4.1 Sanity check: 10 clean files, read every transcript
sample = test_set[test_set["condition"] == "clean"]
sample = (sample if len(sample) else test_set).drop_duplicates("utt_id").head(10)
report = check_outputs(run_predictions(model, sample, "zero-shot", desc="sanity"))
for r in report.itertuples():
    print(f"[{r.utt_id}]\n  ref: {r.text}\n  hyp: {r.hyp}\n")
display(report[["utt_id", "diacritics", "odd_chars", "special_token", "empty"]])
problems = report[~report["diacritics"] | report["odd_chars"].ne("") | report["special_token"] | report["empty"]]
print("OK" if problems.empty else f"CHECK {len(problems)} transcripts above before the full run")

sanity:   0%|          | 0/1 [00:00<?, ?it/s]

[fleurs_10072308967409148015]
  ref: đối với springboks trận này đã giúp đội tuyển kết thúc chuỗi thua năm trận liền
  hyp: Đối với Spring Boot, trận này đã giúp đội tuyển kết thúc chuỗi thua năm trận liền.

[fleurs_10185666587391780903]
  ref: các nguyên tố như canxi và kali được coi là kim loại dĩ nhiên cũng có các kim loại như vàng và bạc
  hyp: Các nguyên tố như canxi và kali được coi là kim loại, dĩ nhiên cũng có các kim loại như vàng và bạc.

[fleurs_10078277432407694754]
  ref: nó cũng tấn công mọi thứ trong nước ngay cả khủng long khổng lồ như t rex cũng không phải là đối thủ với nó
  hyp: nó cũng tấn công mọi thứ trong nước ngay cả khủng long khổng lồ như tigre cũng không phải là đối thủ với nó

[fleurs_1015563747297542961]
  ref: mọi danh từ kế bên từ sie dành cho bạn luôn bắt đầu bằng một chữ viết hoa ngay cả khi ở giữa câu
  hyp: Mỗi danh từ kế bên từ C dành cho bạn luôn bắt đầu bằng một chữ viết hoa ngay cả khi ở dưới câu.

[fleurs_10093159547857080890]
  ref: số người có 

,utt_id,diacritics,odd_chars,special_token,empty
0,fleurs_10072308967409148015,True,,False,False
1,fleurs_10185666587391780903,True,,False,False
2,fleurs_10078277432407694754,True,,False,False
3,fleurs_1015563747297542961,True,,False,False
4,fleurs_10093159547857080890,True,,False,False
5,fleurs_10057024225045006889,True,,False,False
6,fleurs_10188670312101823236,True,,False,False
7,fleurs_1015279164912843543,True,,False,False
8,fleurs_1020360004532126324,True,,False,False
9,fleurs_10169179696365895477,True,,False,False


OK


In [10]:
# 4.2 Full zero-shot test (reloads the CSV if it already exists)
pred_zs = run_or_load(model, test_set, ZS_VARIANT)
pred_zs.sample(5, random_state=SEED)[["condition", "text", "hyp"]]

zero-shot:   0%|          | 0/479 [00:00<?, ?it/s]

wrote /kaggle/working/predictions_qwen3-asr-0.6b_zero-shot.csv (7650 rows)


,condition,text,hyp
2337,babble_10dB,nó nổi tiếng là tiêu điểm của sự xa hoa bắt đầu từ khoảng năm bốn trăm tcn và kéo dài đến tận khoảng năm một nghìn m...,nó nổi tiếng là tiêu điểm của sự xa hoa bắt đầu từ khoảng năm 400 chiếc công nguyên và kéo dài đến tận khoảng năm 1....
5752,traffic_20dB,châu âu là lục địa tương đối nhỏ nhưng có nhiều quốc gia độc lập thông thường việc đi qua nhiều quốc gia đồng nghĩa ...,"Trâu Âu là lục địa tương đối nhỏ nhưng có nhiều quốc gia độc lập. Thông thường, việc đi qua nhiều quốc gia đồng nghĩ..."
2465,babble_0dB,đây sẽ không phải là từ biệt đây chỉ là kết thúc một chương và mở ra một chương mới,đây sẽ không phải là từ bị phá hủy đây chỉ là sơ cấp và chương mở ra
5686,babble_5dB,khu vực này được cai trị bởi chính phủ pháp vichy đây là những người pháp đã làm hòa với người đức vào năm một nghìn...,Khu vực này được cai trị bởi chính thuật pháp viết hy. Đây là những người pháp đã làm hòa với người Đức vào năm 1900...
4747,traffic_0dB,ottawa là thủ đô song ngữ quyến rũ của canada sở hữu hàng loạt phòng trưng bày nghệ thuật và bảo tàng trưng bày các ...,"Ottawa là thủ đô sông Ngũ Quyền lũ của Canada, sở hữu hàng loạt phòng trưng bày nghệ thuật và bảo tàng trưng bày các..."


## 5. Evaluate

WER / CER per noise type × SNR, split into substitutions, deletions and insertions;
95% bootstrap intervals; the worst utterances.

In [11]:
# 5.1 WER / CER per noise × SNR, error split, bootstrap intervals, worst utterances
scored_zs, summary_zs = evaluate(pred_zs, test_set)
print("WER (%)"); display(wer_grid(summary_zs))
print("CER (%)"); display(wer_grid(summary_zs, "cer"))
display(summary_zs[["noise_type", "snr_db", "wer", "cer", "sub_pct", "del_pct", "ins_pct"]])
display(bootstrap_ci(scored_zs))
display(worst(scored_zs))

WER (%)


snr                                          clean  20 dB  10 dB   5 dB    0 dB
model          variant   dataset noise_type                                    
qwen3-asr-0.6b zero-shot fleurs  babble        NaN   8.66  13.28  30.65  109.59
                                 clean        8.33    NaN    NaN    NaN     NaN
                                 traffic       NaN   8.81  10.09  12.87   19.73

CER (%)


snr                                          clean  20 dB  10 dB   5 dB   0 dB
model          variant   dataset noise_type                                   
qwen3-asr-0.6b zero-shot fleurs  babble        NaN   4.62   8.09  21.40  87.82
                                 clean        4.39    NaN    NaN    NaN    NaN
                                 traffic       NaN   4.73   5.54   7.49  12.40

,noise_type,snr_db,wer,cer,sub_pct,del_pct,ins_pct
0,babble,0.0,109.59,87.82,55.50,18.90,35.19
1,babble,5.0,30.65,21.40,21.16,3.47,6.02
2,babble,10.0,13.28,8.09,10.27,0.95,2.07
3,babble,20.0,8.66,4.62,7.14,0.62,0.90
4,clean,NaN,8.33,4.39,6.86,0.61,0.86
5,traffic,0.0,19.73,12.40,16.42,2.02,1.29
6,traffic,5.0,12.87,7.49,10.73,1.03,1.11
7,traffic,10.0,10.09,5.54,8.27,0.83,0.99
8,traffic,20.0,8.81,4.73,7.24,0.67,0.90


,model,variant,dataset,noise_type,snr_db,wer,ci_low,ci_high
0,qwen3-asr-0.6b,zero-shot,fleurs,babble,0.0,109.59,98.58,121.95
1,qwen3-asr-0.6b,zero-shot,fleurs,babble,5.0,30.65,26.33,36.12
2,qwen3-asr-0.6b,zero-shot,fleurs,babble,10.0,13.28,11.74,15.92
3,qwen3-asr-0.6b,zero-shot,fleurs,babble,20.0,8.66,8.10,9.23
4,qwen3-asr-0.6b,zero-shot,fleurs,clean,NaN,8.33,7.77,8.89
5,qwen3-asr-0.6b,zero-shot,fleurs,traffic,0.0,19.73,18.53,21.06
6,qwen3-asr-0.6b,zero-shot,fleurs,traffic,5.0,12.87,12.03,13.71
7,qwen3-asr-0.6b,zero-shot,fleurs,traffic,10.0,10.09,9.46,10.79
8,qwen3-asr-0.6b,zero-shot,fleurs,traffic,20.0,8.81,8.22,9.42


,utt_id,condition,utt_wer,ref,hyp_norm
4678,fleurs_5112472991553410708,babble_5dB,1995.0,giancarlo fisichella mất kiểm soát chiếc xe của mình và kết thúc cuộc đua rất sớm sau khi bắt đầu,cái này thì khi mà cái này là cái gì thì cái này là cái gì thì cái này là cái gì thì cái này là cái gì thì cái này l...
2159,fleurs_14619898958407725934,babble_0dB,1813.6,trong hai năm kết thúc cuộc chiến các đồng minh trước kia lại là kẻ thù và chiến tranh lạnh bắt đầu,bạn nói chuyện với người ở malaysia thì người ở malaysia thì người ở malaysia thì người ở malaysia thì người ở malay...
5363,fleurs_17238531885163339652,babble_0dB,1721.7,chỉ huy cảnh sát chandra shekhar solanki nói rằng những bị cáo đã xuất hiện tại tòa với khuôn mặt bị che kín,tôi nghĩ rằng chúng tôi đã thực hiện một điều rất rõ ràng là tôi đã hiểu rằng điều đó là một sự sai lầm lớn và tôi đ...
5849,fleurs_16168406173655143575,babble_0dB,1274.2,quần đảo nằm cách bán đảo một trăm hai mươi ki lô mét phía bắc peninsula đảo lớn nhất là king george với sự hiện diệ...,cái đó là một cái con dao như cái con dao của người dân là một cái con dao lớn như cái con dao của người dân là một ...
7297,fleurs_6636546545430971862,babble_5dB,1239.1,thống đốc bang california arnold schwarzenegger đã ký thành luật cấm bán hoặc cho thuê trò chơi video mang tính chất...,không có bao nhiêu kilometer các thành phố này các series các phim hoạt hình các bộ phim hoạt hình các bộ phim hoạt ...


## 6. Fine-tuning setup

`load_train_set` serves the train utterances of `DATASETS`. `ft-noisy` mixes traffic or
babble from the train noise pool on the fly, from the row of `train_schedule.csv` for the
current epoch (25% clean, SNR 0–20 dB): each epoch gets new noise and every model gets the
same samples. `ft-clean` serves clean audio. LoRA is added to the layers of the recipe
(`targets`; with `LORA_ALL`, every attention and MLP projection of the speech encoder and of
the Qwen3 language model); the base weights are frozen. Cell 6.1 prints how many layers of each
part got an adapter.

**One training row** is the test-time prompt followed by the target:

```
[padding] <prompt with the audio> language Vietnamese<asr_text> | transcript <|im_end|> [padding]
          └──────────── no loss (label -100) ─────────────────┘ └──── loss ─────────┘
```

The prompt comes from the same `apply_transcription_request` call that `transcribe` uses, so
training and testing see the same input. The vocabulary has about 150,000 tokens, so logits
are computed only for the target columns (`logits_to_keep`): a 30 s utterance has about 390
audio positions whose logits would never be used.

**Length-grouped batches** (`GROUP_BY_LENGTH`). The speech encoder packs the real frames and
skips the padding, but the language model runs over the whole padded row: a 3 s utterance in a
batch with a 30 s one carries about 350 padding positions through every layer. Each epoch, the
training utterances are shuffled, cut into groups of 50 batches, sorted by duration inside each
group and cut into batches; the batches are then shuffled. Batches change every epoch but hold
utterances of similar length.

**SpecAugment** (recipes with `augment=True`, training only): in each utterance, zero
`SPEC_FREQ_MASKS` bands of up to `SPEC_FREQ_WIDTH` mel bins and `SPEC_TIME_MASKS` spans of up to
`SPEC_TIME_FRACTION` of its frames. The padded frames are left alone.

**Encoder-focused rank** (recipes with `rank_pattern`). `peft` gives every targeted layer whose
name matches `ENCODER_PATTERN` (`audio_tower.*`: the speech encoder) the rank and alpha of
`rank_pattern` / `alpha_pattern`, and every other targeted layer (the language model) the
recipe's `r` / `alpha`. `add_lora` reads the rank of every adapted layer back and stops with an
error if they are not the ones the recipe asks for.

**Clean/noisy consistency loss** (recipes with `consistency` > 0, training only). `mix` keeps the
length of the speech, so a noisy training sample and the clean file it was made from give the
speech encoder the same frames. `collate` loads the clean file again and computes its features;
`train_loss` runs the encoder without adapters (the zero-shot encoder) on them, without
gradient, and adds

    consistency × mean over frames of (1 − cos(adapted encoder on noisy audio, original encoder on clean audio))

to the cross-entropy. Both sides are the encoder + projector output that replaces the audio
tokens in the language model's input; the adapted side is the one the cross-entropy forward
already computes, so the extra cost is one encoder pass without gradient and a second feature
extraction in the DataLoader. The term is in [0, 2] and does not depend on the scale of the
features. On the clean training samples (25%) both sides see the same audio, and the term keeps
the adapted encoder close to the original one.

**Encoder-only LoRA** (`ENCODER_ONLY`). `targets` may also be a regular expression, which `peft`
matches against the whole module name inside the base model: `ENCODER_ONLY_TARGETS` selects the
attention and MLP projections of the speech encoder, so the language model keeps its weights.

**Restart check.** After an error such as running out of GPU memory, Python keeps the failed cell's
tensors alive for the traceback, and running cells again loads a second model on top. Cell 6.1 stops
when the GPU holds clearly more than one model right after loading it: restart the kernel
(*Run → Restart & clear cell outputs*) and run again from 0.1.

In [12]:
# 6.1 Training set (online noise), LoRA model with per-part ranks, collate (+ clean features for the
# consistency loss), train_loss, SpecAugment, length-grouped batches
# Kaggle ships an old torchao that recent peft rejects; we do not use torchao.
import peft.import_utils
import peft.tuners.lora.torchao
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False
import json
from collections import Counter
from functools import partial

import torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from peft.tuners.lora import LoraLayer
from torch.utils.data import DataLoader, Sampler, Subset
from transformers import get_linear_schedule_with_warmup

train_set = load_train_set(M / "clean.csv", DATA_ROOT, VARIANT, schedule=M / "train_schedule.csv",
                           noise_pools=[M / "noise_traffic.csv", M / "noise_voices.csv"],
                           datasets=DATASETS)
durations = clean.set_index("utt_id")["duration_s"]
train_lengths = durations.loc[list(train_set.utt_ids)].to_numpy()   # seconds, in train_set order
clean_paths = clean.set_index("utt_id")["path"]      # the clean file a training sample is made from
print(f"{VARIANT}: {len(train_set)} training utterances, {train_lengths.sum() / 3600:.1f} h")

tokenizer = processor.tokenizer
EOS = tokenizer.eos_token                  # "<|im_end|>": ends the answer, generation stops on it

for name in ["model", "ft_model", "best_model", "base"]:  # free GPU memory from earlier steps
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()


def model_part(name):
    return ("speech encoder" if ".audio_tower." in name else
            "projector" if ".multi_modal_projector." in name else "language model")


def lora_layers(model):
    # How many layers of each kind carry an adapter, per part of the model
    parts = {}
    for name, module in model.named_modules():
        if isinstance(module, LoraLayer):
            parts.setdefault(model_part(name), Counter())[name.rsplit(".", 1)[-1]] += 1
    return {part: dict(kinds) for part, kinds in parts.items()}


def lora_ranks(model):
    # The ranks of the adapted layers, per part of the model
    ranks = {}
    for name, module in model.named_modules():
        if isinstance(module, LoraLayer):
            ranks.setdefault(model_part(name), set()).add(module.r["default"])
    return ranks


def add_lora(base, run=RUN):
    # A fresh adapter for one recipe, with the same initialization every time. Stops when the
    # adapted layers do not have the ranks the recipe asks for (e.g. a pattern that matches nothing).
    torch.manual_seed(SEED)
    model = get_peft_model(base, LoraConfig(r=run["r"], lora_alpha=run["alpha"], lora_dropout=LORA_DROPOUT,
                                            target_modules=run["targets"], bias="none",
                                            rank_pattern=run.get("rank_pattern", {}),
                                            alpha_pattern=run.get("alpha_pattern", {})))
    ranks = lora_ranks(model)
    encoder_ranks = set(run.get("rank_pattern", {}).values()) or {run["r"]}
    expected = {part: encoder_ranks if part == "speech encoder" else {run["r"]} for part in ranks}
    if ranks != expected or (run.get("rank_pattern") and "speech encoder" not in ranks):
        raise RuntimeError(f"recipe {run['name']!r}: LoRA ranks per part are {ranks}, expected {expected}; "
                           "check ENCODER_PATTERN and the recipe's targets against the module names")
    return model


base = load_base_model()
if DEVICE == "cuda":
    # One model in float32 takes about 3.2 GB; much more means memory held by an earlier failed cell
    held = torch.cuda.memory_allocated() / 2**30
    print(f"GPU memory after loading the model: {held:.1f} GB")
    if held > 5:
        raise RuntimeError(f"{held:.1f} GB in use after loading one model: an earlier failed cell still holds GPU "
                           "memory. Restart the kernel (Run -> Restart & clear cell outputs) and run again from 0.1.")
if GRAD_CHECKPOINT:
    base.gradient_checkpointing_enable()
ft_model = add_lora(base, RUN)
ft_model.print_trainable_parameters()
print(lora_layers(ft_model))
print("LoRA ranks:", lora_ranks(ft_model))


def spec_augment(features, mask):
    # features (batch, 128 mel bins, frames), mask (batch, frames) with 1 on real frames.
    # Real frames come first (the feature extractor pads on the right); padding is left alone.
    # Masked values are set to 0, as the transformers Whisper implementation does.
    features = features.clone()
    n_mels = features.shape[1]
    for i, n_frames in enumerate(mask.sum(-1).long().tolist()):
        for _ in range(SPEC_FREQ_MASKS):
            width = int(torch.randint(0, SPEC_FREQ_WIDTH + 1, ()))
            start = int(torch.randint(0, n_mels - width + 1, ()))
            features[i, start:start + width, :n_frames] = 0
        max_width = int(SPEC_TIME_FRACTION * n_frames)
        for _ in range(SPEC_TIME_MASKS):
            width = int(torch.randint(0, max_width + 1, ()))
            start = int(torch.randint(0, n_frames - width + 1, ()))
            features[i, :, start:start + width] = 0
    return features


def collate(items, augment=False, consistency=False):
    batch = collate_raw(items)             # utt_id, audio, text, noise_type, snr_db (lists)
    # Prompt: the one transcribe() uses, left-padded, ending with "language Vietnamese<asr_text>"
    prompt = processor.apply_transcription_request(audio=batch["audio"], language=LANGUAGE)
    features = prompt["input_features"]
    if augment:                            # training only: 6.2, 6.3 and dev / test never augment
        features = spec_augment(features, prompt["input_features_mask"])
    # Target: transcript + end-of-turn token, right-padded
    target = tokenizer([text + EOS for text in batch["text"]], add_special_tokens=False,
                       padding=True, padding_side="right", truncation=True,
                       max_length=MAX_LABEL_TOKENS, return_tensors="pt")
    labels = target.input_ids.masked_fill(target.attention_mask.eq(0), -100)   # ignore padding
    out = {
        "input_ids": torch.cat([prompt["input_ids"], target.input_ids], dim=1),
        "attention_mask": torch.cat([prompt["attention_mask"], target.attention_mask], dim=1),
        "input_features": features,
        "input_features_mask": prompt["input_features_mask"],
        # Logits are kept for the target columns plus the column before them (the last prompt
        # token, which predicts the first target token); labels cover the same columns.
        "labels": F.pad(labels, (1, 0), value=-100),
        "logits_to_keep": labels.shape[1] + 1,
    }
    if consistency:
        # The clean files the samples were made from, in the same order and never augmented. mix()
        # keeps the length of the speech, so the frames match those of `features` one to one.
        clean_audio = [load_audio(DATA_ROOT / clean_paths.loc[u]) for u in batch["utt_id"]]
        clean_prompt = processor.apply_transcription_request(audio=clean_audio, language=LANGUAGE)
        out["clean_input_features"] = clean_prompt["input_features"]
        out["clean_input_features_mask"] = clean_prompt["input_features_mask"]
    return out


@torch.no_grad()
def teacher_audio_features(model, features, mask):
    # Encoder + projector output of the model without adapters (the zero-shot model), in eval mode
    was_training = model.training
    model.eval()
    try:
        with model.disable_adapter(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP):
            return model.get_audio_features(input_features=features, input_features_mask=mask,
                                            return_dict=True).pooler_output
    finally:
        model.train(was_training)


def train_loss(model, batch, consistency_weight=0.0):
    """Loss of one batch: cross-entropy, plus consistency_weight × the clean/noisy consistency loss.

    Returns (loss, parts) with parts = {"asr_loss": float, "consistency_loss": float or None}.
    The consistency loss needs the clean features of collate(..., consistency=True).
    """
    batch = dict(batch)                    # the caller's batch is left as it is
    clean_features = batch.pop("clean_input_features", None)
    clean_mask = batch.pop("clean_input_features_mask", None)
    batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP):
        outputs = model(**batch)
    loss = outputs.loss
    parts = {"asr_loss": loss.item(), "consistency_loss": None}
    if consistency_weight and clean_features is not None:
        teacher = teacher_audio_features(model, clean_features.to(DEVICE), clean_mask.to(DEVICE))
        student = outputs.audio_hidden_states  # adapted encoder + projector on the noisy audio, with gradient
        if student.shape != teacher.shape:
            raise RuntimeError(f"consistency loss: noisy features {tuple(student.shape)} and clean features "
                               f"{tuple(teacher.shape)} differ; the clean and noisy audio must have the same length")
        consistency = 1 - F.cosine_similarity(student.float(), teacher.float(), dim=-1).mean()
        loss = loss + consistency_weight * consistency
        parts["consistency_loss"] = consistency.item()
    return loss, parts


class LengthGroupedBatches(Sampler):
    """Batches of train_set[indices] whose utterances have similar durations, new every epoch.

    Shuffles the indices, cuts them into groups of `group_batches` batches, sorts each group by
    duration, cuts it into batches and shuffles the batches. The order depends on seed + epoch
    only, so every recipe and every model sees the same batches.
    """

    def __init__(self, lengths, batch_size, indices, seed=SEED, group_batches=50):
        self.lengths, self.indices = np.asarray(lengths), np.asarray(indices)
        self.batch_size, self.group = batch_size, batch_size * group_batches
        self.seed, self.epoch = seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return math.ceil(len(self.indices) / self.batch_size)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        order = rng.permutation(self.indices)
        batches = []
        for start in range(0, len(order), self.group):
            group = order[start:start + self.group]
            group = group[np.argsort(-self.lengths[group], kind="stable")]
            batches += [group[i:i + self.batch_size].tolist() for i in range(0, len(group), self.batch_size)]
        return iter([batches[i] for i in rng.permutation(len(batches))])


def make_loader(indices, epoch, run):
    # One epoch over train_set[indices]: new noise for this epoch, the same for every model
    train_set.set_epoch(epoch)
    collate_fn = partial(collate, augment=run["augment"], consistency=bool(run.get("consistency", 0)))
    common = dict(collate_fn=collate_fn, num_workers=NUM_WORKERS,
                  generator=torch.Generator().manual_seed(SEED + epoch))   # also seeds the workers
    if GROUP_BY_LENGTH:
        sampler = LengthGroupedBatches(train_lengths, TRAIN_BATCH, indices)
        sampler.set_epoch(epoch)
        return DataLoader(train_set, batch_sampler=sampler, **common)
    return DataLoader(Subset(train_set, [int(i) for i in indices]), batch_size=TRAIN_BATCH, shuffle=True, **common)


def new_scaler():
    return torch.amp.GradScaler("cuda", enabled=AMP and AMP_DTYPE == torch.float16)

ft-noisy: 2990 training utterances, 9.0 h


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

GPU memory after loading the model: 2.9 GB
trainable params: 14,336,000 || all params: 796,762,112 || trainable%: 1.7993
{'speech encoder': {'k_proj': 18, 'v_proj': 18, 'q_proj': 18, 'out_proj': 18, 'fc1': 18, 'fc2': 18}, 'language model': {'q_proj': 28, 'k_proj': 28, 'v_proj': 28, 'o_proj': 28, 'gate_proj': 28, 'up_proj': 28, 'down_proj': 28}}
LoRA ranks: {'speech encoder': {32}, 'language model': {8}}


### Checks before training

- **6.2** prints one training row (prompt and target), compares it with the training format
  in the `transformers` documentation, and measures GPU memory on the longest batch. If that
  batch runs out of memory, set `GRAD_CHECKPOINT = True` or halve `TRAIN_BATCH` and double
  `GRAD_ACCUM`, then rerun from 6.1.
- **6.3** trains on 16 utterances only. The loss must fall steadily; a flat or rising loss
  means the pipeline is wrong. The adapter of this check is thrown away afterwards. This cell
  can be skipped.

In [13]:
# 6.2 Check one batch, listen to one training sample, measure memory on the longest batch
train_set.set_epoch(1)
item = train_set[0]
print(item["noise_type"], item["snr_db"], "|", item["text"])
display(Audio(item["audio"], rate=SR))

items = [train_set[i] for i in range(4)]
batch = collate(items)
with torch.no_grad():
    loss, _ = train_loss(ft_model, batch)
print({k: tuple(v.shape) if torch.is_tensor(v) else v for k, v in batch.items()}, f"loss {loss.item():.3f}")

row = batch["input_ids"][0][batch["attention_mask"][0].bool()]            # row 0 without padding
n_row_target = int((batch["labels"][0] != -100).sum())
shorten = lambda text: re.sub(r"(<\|audio_pad\|>)+", lambda m: f"<|audio_pad|>×{m.group(0).count('<|audio_pad|>')}", text)
print("prompt:", repr(shorten(tokenizer.decode(row[:-n_row_target]))))
print("target:", repr(tokenizer.decode(batch["labels"][0][batch["labels"][0] != -100])))

# The same row written the way the transformers documentation trains the model
documented = processor.apply_chat_template(
    [[{"role": "user", "content": [{"type": "audio", "audio": items[0]["audio"]}]},
      {"role": "assistant", "content": [{"type": "text", "text": f"language {LANGUAGE}<asr_text>{items[0]['text']}"}]}]],
    tokenize=True, return_dict=True)["input_ids"][0].tolist()
assert documented[:len(row)] == row.tolist(), "the training row differs from the documented training format"
print("matches the documented format; tokens it adds after ours:", repr(tokenizer.decode(documented[len(row):])))

# Worst case for memory: the TRAIN_BATCH longest training utterances, one forward + backward
# (with GROUP_BY_LENGTH, training really meets this batch), with the consistency loss of RUN
longest = np.argsort(-train_lengths)[:TRAIN_BATCH]
batch = collate([train_set[int(i)] for i in longest], augment=RUN["augment"],
                consistency=bool(RUN.get("consistency", 0)))
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
ft_model.train()
loss, parts = train_loss(ft_model, batch, consistency_weight=RUN.get("consistency", 0.0))
loss.backward()
print("losses on the longest batch:", parts)
ft_model.zero_grad(set_to_none=True)
ft_model.eval()
peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == "cuda" else float("nan")
print(f"longest batch: {train_lengths[longest[0]]:.1f} s audio, "
      f"{batch['input_ids'].shape[1]} tokens, {batch['labels'].shape[1]} with logits, peak GPU memory {peak:.1f} GB")

babble 17.17 | theo sau quá trình tố tụng này hjr ba sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm hai nghìn không trăm mười lăm hoặc hai nghìn không trăm mười sáu để tiếp tục quá trình xử lý


{'input_ids': (4, 304), 'attention_mask': (4, 304), 'input_features': (4, 128, 1800), 'input_features_mask': (4, 1800), 'labels': (4, 57), 'logits_to_keep': 57} loss 0.659
prompt: '<|im_start|>system\n<|im_end|>\n<|im_start|>user\n<|audio_start|><|audio_pad|>×187<|audio_end|><|im_end|>\n<|im_start|>assistant\nlanguage Vietnamese<asr_text>'
target: 'theo sau quá trình tố tụng này hjr ba sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm hai nghìn không trăm mười lăm hoặc hai nghìn không trăm mười sáu để tiếp tục quá trình xử lý<|im_end|>'
matches the documented format; tokens it adds after ours: '\n'


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


losses on the longest batch: {'asr_loss': 0.4822126626968384, 'consistency_loss': 0.05774867534637451}
longest batch: 26.8 s audio, 464 tokens, 99 with logits, peak GPU memory 5.5 GB


In [14]:
# 6.3 Overfit check: 16 utterances, 40 updates. The loss must fall steadily
OVERFIT_UTTS, OVERFIT_STEPS, OVERFIT_LR = 16, 40, 1e-3
items = [train_set[i] for i in range(OVERFIT_UTTS)]
optimizer = torch.optim.AdamW([p for p in ft_model.parameters() if p.requires_grad], lr=OVERFIT_LR)
scaler = new_scaler()
ft_model.train()
trace = []
for step in range(OVERFIT_STEPS):
    start = (step * TRAIN_BATCH) % OVERFIT_UTTS
    batch = collate(items[start:start + TRAIN_BATCH], augment=RUN["augment"],
                    consistency=bool(RUN.get("consistency", 0)))
    loss, parts = train_loss(ft_model, batch, consistency_weight=RUN.get("consistency", 0.0))
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    trace.append(loss.item())
ft_model.eval()
print("loss every 5 updates:", " ".join(f"{v:.3f}" for v in trace[::5] + trace[-1:]))
print("last update:", parts)
print("ref:", items[0]["text"])
print("hyp:", transcribe(ft_model, processor, [items[0]["audio"]])[0])

# Throw this adapter away: step 7 starts from a fresh one on the untouched base weights
ft_model = add_lora(ft_model.unload(), RUN)
del optimizer, scaler
gc.collect(); torch.cuda.empty_cache()

loss every 5 updates: 0.723 0.712 0.569 0.185 0.194 0.171 0.120 0.103 0.096
last update: {'asr_loss': 0.0012797459494322538, 'consistency_loss': 0.09456384181976318}
ref: theo sau quá trình tố tụng này hjr ba sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm hai nghìn không trăm mười lăm hoặc hai nghìn không trăm mười sáu để tiếp tục quá trình xử lý
hyp: theo sau quá trình tố tụng này hjr ba sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm hai nghìn không trăm mười lăm hoặc hai nghìn không trăm mười sáu để tiếp tục quá trình xử lý


## 7. Fine-tuning

**7.0** defines `train_lora(model, run, indices, epochs)`. Dev WER is measured on `dev_fast` (the
whole dev set by default) before training and after each epoch. Epoch 0 is the zero-shot model,
because a fresh LoRA adapter changes nothing; it is computed once, saved in
`dev_predictions_*_zero-shot*.csv` and reused. For dev decoding the adapter is merged into the
weights, then taken out again. The adapter with the lowest dev WER is saved, and training stops
after `PATIENCE` epochs without a lower dev WER. The test set is not used here.

**7.1** trains every recipe of `CANDIDATES` for `EPOCHS` on `TRAIN_FRACTION` of the training set,
each from a fresh adapter on the untouched base weights, and keeps each one's best adapter in
`lora_*_<variant>-<tag>/`. A finished candidate leaves `candidate_*.json`; when 7.1 runs again in
the same session (for example after restarting the kernel because of an error), a candidate whose
settings did not change is not trained again. **7.2** picks the candidate with the lowest dev WER
and writes `run_*.json`, which step 8 reads.

One update uses `GRAD_ACCUM` batches of `TRAIN_BATCH` utterances. A batch whose loss is not
finite (float16 overflow) is left out and counted in the `skipped` column; the column should
stay at 0. If it does not, set `MIXED_PRECISION = False` and rerun. The history also splits
`train_loss` into `asr_loss` (cross-entropy) and `consistency_loss`. A training loss that keeps
falling while the dev WER does not means over-fitting, as in 03c with lr 2e-4.

In [15]:
# 7.0 Training function: dev WER before training and after each epoch, best adapter, early stopping
def dev_wer(model, epoch):
    if epoch == 0:
        # A fresh adapter adds 0 (LoRA starts with B = 0): this is the zero-shot model, computed once
        pred = run_or_load(model, dev_fast, ZS_VARIANT, path=OUT / f"dev_predictions_{MODEL}_{ZS_VARIANT}.csv")
    else:
        model.merge_adapter()              # faster decoding; taken out again before training goes on
        try:
            pred = run_predictions(model, dev_fast, VARIANT, desc=f"dev, epoch {epoch}")
        finally:
            model.unmerge_adapter()
    _, summary = evaluate(pred, dev_fast, by=("model", "variant"))
    return float(summary["wer"].iloc[0])


def train_lora(model, run, indices, epochs, adapter_dir=None, history_path=None):
    """Train the adapter of `model` in place on train_set[indices]; return the history.

    Dev WER on dev_fast before training and after each epoch. With `adapter_dir`, the adapter
    with the lowest dev WER is saved there. Stops after PATIENCE epochs without a lower dev WER.
    The history splits the training loss into asr_loss and consistency_loss (NaN when off).
    """
    updates_per_epoch = math.ceil(math.ceil(len(indices) / TRAIN_BATCH) / GRAD_ACCUM)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=run["lr"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * epochs * updates_per_epoch),
                                                epochs * updates_per_epoch)
    scaler = new_scaler()
    weight = run.get("consistency", 0.0)
    torch.manual_seed(SEED)

    model.eval()
    history = [{"epoch": 0, "train_loss": None, "asr_loss": None, "consistency_loss": None,
                "skipped": 0, "dev_wer": dev_wer(model, 0)}]
    print(history[-1])
    best_wer, best_epoch = math.inf, 0
    for epoch in range(1, epochs + 1):
        loader = make_loader(indices, epoch, run)
        model.train()
        losses, asr_losses, cons_losses, skipped, pending = [], [], [], 0, 0
        for step, batch in enumerate(tqdm(loader, desc=f"{run['name']} | epoch {epoch}"), start=1):
            loss, parts = train_loss(model, batch, consistency_weight=weight)
            if torch.isfinite(loss):
                scaler.scale(loss / GRAD_ACCUM).backward()
                losses.append(loss.item())
                asr_losses.append(parts["asr_loss"])
                if parts["consistency_loss"] is not None:
                    cons_losses.append(parts["consistency_loss"])
                pending += 1
            else:
                skipped += 1
            if step % GRAD_ACCUM == 0 or step == len(loader):
                if pending:                # GradScaler fails on an update without any gradient
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                pending = 0
        model.eval()
        wer = dev_wer(model, epoch)
        mean = lambda values: float(np.mean(values)) if values else math.nan
        history.append({"epoch": epoch, "train_loss": mean(losses), "asr_loss": mean(asr_losses),
                        "consistency_loss": mean(cons_losses), "skipped": skipped, "dev_wer": wer})
        improved = wer < best_wer
        if improved:
            best_wer, best_epoch = wer, epoch
            if adapter_dir is not None:
                model.save_pretrained(adapter_dir)
        print(history[-1], "(best)" if improved else "")
        if history_path is not None:
            pd.DataFrame(history).to_csv(history_path, index=False)
        if epoch - best_epoch >= PATIENCE:
            print(f"early stop: no lower dev WER for {PATIENCE} epochs, best epoch {best_epoch}")
            break
    return pd.DataFrame(history)

In [16]:
# 7.1 Train every candidate recipe, keep each one's best adapter (lowest dev WER)
n_train = int(round(TRAIN_FRACTION * len(train_set)))
train_idx = (np.arange(len(train_set)) if n_train >= len(train_set) else
             np.sort(np.random.default_rng(SEED).permutation(len(train_set))[:n_train]))
results = []
for run in CANDIDATES:
    name = f"{VARIANT}-{run['tag']}"
    adapter_dir = OUT / f"lora_{MODEL}_{name}"
    done_path = OUT / f"candidate_{MODEL}_{name}{DECODE_TAG}.json"
    # A finished candidate is reused only if it was trained and selected with these settings
    settings = {"recipe": run, "epochs": EPOCHS, "patience": PATIENCE, "train_utterances": len(train_idx),
                "train_batch": TRAIN_BATCH, "grad_accum": GRAD_ACCUM, "dev_files": len(dev_fast),
                "decode": DECODE_TAG}
    done = json.loads(done_path.read_text(encoding="utf-8")) if done_path.exists() else None
    if done is not None and done["settings"] == settings and adapter_dir.exists():
        print(f"{run['name']}: already trained, dev WER {done['dev_wer']} at epoch {done['best_epoch']}")
        results.append(done)
        continue
    ft_model = add_lora(ft_model.unload(), run)          # fresh adapter on the untouched base weights
    print(run["name"], lora_layers(ft_model), "ranks:", lora_ranks(ft_model))
    ft_model.print_trainable_parameters()
    hist = train_lora(ft_model, run, train_idx, EPOCHS, adapter_dir=adapter_dir,
                      history_path=OUT / f"history_{MODEL}_{name}{DECODE_TAG}.csv")
    best = hist.iloc[1:]["dev_wer"].idxmin()
    done = {"name": run["name"], "tag": run["tag"], "adapter_dir": str(adapter_dir),
            "dev_wer_zero_shot": float(hist["dev_wer"].iloc[0]), "dev_wer": float(hist.loc[best, "dev_wer"]),
            "best_epoch": int(hist.loc[best, "epoch"]), "epochs_run": int(hist["epoch"].iloc[-1]),
            "trainable_M": round(sum(p.numel() for p in ft_model.parameters() if p.requires_grad) / 1e6, 2),
            "settings": settings}
    done_path.write_text(json.dumps(done, indent=2, ensure_ascii=False), encoding="utf-8")
    results.append(done)
    gc.collect(); torch.cuda.empty_cache()

candidates = (pd.DataFrame([{k: v for k, v in r.items() if k not in ("settings", "adapter_dir")} for r in results])
              .sort_values("dev_wer", kind="stable").reset_index(drop=True))
candidates.to_csv(OUT / f"candidates_{MODEL}_{VARIANT}{DECODE_TAG}.csv", index=False)
candidates

encoder r32 / LM r8 + consistency, lr 1e-4 {'speech encoder': {'k_proj': 18, 'v_proj': 18, 'q_proj': 18, 'out_proj': 18, 'fc1': 18, 'fc2': 18}, 'language model': {'q_proj': 28, 'k_proj': 28, 'v_proj': 28, 'o_proj': 28, 'gate_proj': 28, 'up_proj': 28, 'down_proj': 28}} ranks: {'speech encoder': {32}, 'language model': {8}}
trainable params: 14,336,000 || all params: 796,762,112 || trainable%: 1.7993


zero-shot:   0%|          | 0/23 [00:00<?, ?it/s]

wrote /kaggle/working/dev_predictions_qwen3-asr-0.6b_zero-shot.csv (359 rows)
{'epoch': 0, 'train_loss': None, 'asr_loss': None, 'consistency_loss': None, 'skipped': 0, 'dev_wer': 23.22}


encoder r32 / LM r8 + consistency, lr 1e-4 | epoch 1:   0%|          | 0/748 [00:00<?, ?it/s]

dev, epoch 1:   0%|          | 0/23 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.3430018295698306, 'asr_loss': 0.2712749711469433, 'consistency_loss': 0.07172685805807777, 'skipped': 0, 'dev_wer': 15.47} (best)


... (30 outputs left out)

encoder r32 / LM r8 + consistency, lr 2e-4 (03c recipe) | epoch 3:   0%|          | 0/748 [00:00<?, ?it/s]

dev, epoch 3:   0%|          | 0/23 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.13891317052169042, 'asr_loss': 0.08844684867576177, 'consistency_loss': 0.050466321448591304, 'skipped': 0, 'dev_wer': 15.63} (best)


,name,tag,dev_wer_zero_shot,dev_wer,best_epoch,epochs_run,trainable_M
0,"encoder r32 / LM r8 + consistency, lr 1e-4",enc32-lm8-cons-lr1e-4,23.22,15.22,3,3,14.34
1,"encoder r32 / LM r8 + consistency, lr 5e-5",enc32-lm8-cons-lr5e-5,23.22,15.33,2,3,14.34
2,"encoder-only r32 + consistency, lr 1e-4",enc32-only-cons-lr1e-4,23.22,15.59,3,3,9.29
3,"encoder r32 / LM r8 + consistency, lr 2e-4 (03c recipe)",enc32-lm8-cons-lr2e-4,23.22,15.63,3,3,14.34


In [17]:
# 7.2 Choose the candidate with the lowest dev WER; step 8 tests its adapter
chosen = next(r for r in results if r["tag"] == candidates["tag"].iloc[0])
(OUT / f"run_{MODEL}_{VARIANT}{DECODE_TAG}.json").write_text(json.dumps(chosen, indent=2, ensure_ascii=False),
                                                            encoding="utf-8")
test_variant = f"{VARIANT}-{chosen['tag']}{DECODE_TAG}"
print(f"chosen: {chosen['name']} | dev WER {chosen['dev_wer']} at epoch {chosen['best_epoch']} "
      f"(zero-shot {chosen['dev_wer_zero_shot']}) | test predictions: {predictions_path(test_variant).name}")
pd.read_csv(OUT / f"history_{MODEL}_{VARIANT}-{chosen['tag']}{DECODE_TAG}.csv")

chosen: encoder r32 / LM r8 + consistency, lr 1e-4 | dev WER 15.22 at epoch 3 (zero-shot 23.22) | test predictions: predictions_qwen3-asr-0.6b_ft-noisy-enc32-lm8-cons-lr1e-4.csv


,epoch,train_loss,asr_loss,consistency_loss,skipped,dev_wer
0,0,NaN,NaN,NaN,0,23.22
1,1,0.343002,0.271275,0.071727,0,15.47
2,2,0.215171,0.160081,0.055090,0,15.61
3,3,0.166647,0.115560,0.051087,0,15.22


## 8. Test the fine-tuned model

Loads the adapter chosen in 7.2 (`run_*.json`), merges it into the weights, runs the same test table
with the same decoding, and compares it with zero-shot. The fine-tuned predictions are saved as
`predictions_*_<variant>-<tag>[-cap].csv`, one file per recipe, and recomputed when the adapter is
newer than the saved CSV; the zero-shot ones are reloaded from step 4, and the base model is loaded
only when they do not match the test table. `compare` is a paired bootstrap of
`WER(fine-tuned) − WER(zero-shot)`: a negative `diff` means fewer errors; `significant` is True
when the 95% interval excludes 0.

In [18]:
# 8.1 Test the chosen adapter, compare with zero-shot (same decoding)
for name in ["model", "ft_model", "base"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

chosen = json.loads((OUT / f"run_{MODEL}_{VARIANT}{DECODE_TAG}.json").read_text(encoding="utf-8"))
adapter_dir = Path(chosen["adapter_dir"])
test_variant = f"{VARIANT}-{chosen['tag']}{DECODE_TAG}"
print(f"testing {chosen['name']} ({adapter_dir.name}) as variant {test_variant!r}")
best_model = PeftModel.from_pretrained(load_base_model(), adapter_dir).merge_and_unload().to(DEVICE).eval()
adapter_time = max(f.stat().st_mtime for f in adapter_dir.iterdir())
ft_path = predictions_path(test_variant)
pred_ft = run_or_load(best_model, test_set, test_variant,
                      force=ft_path.exists() and ft_path.stat().st_mtime < adapter_time)
pred_zs = run_or_load(load_base_model, test_set, ZS_VARIANT)   # loads the base model only if needed
scored_ft, _ = evaluate(pred_ft, test_set)
scored_zs, _ = evaluate(pred_zs, test_set)

both = pd.concat([scored_zs, scored_ft], ignore_index=True)
print("WER (%)"); display(wer_grid(summarize(both)))
print("CER (%)"); display(wer_grid(summarize(both), "cer"))
display(compare(both, (MODEL, ZS_VARIANT), (MODEL, test_variant)))
display(worst(scored_ft))

testing encoder r32 / LM r8 + consistency, lr 1e-4 (lora_qwen3-asr-0.6b_ft-noisy-enc32-lm8-cons-lr1e-4) as variant 'ft-noisy-enc32-lm8-cons-lr1e-4'


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

ft-noisy-enc32-lm8-cons-lr1e-4:   0%|          | 0/479 [00:00<?, ?it/s]

wrote /kaggle/working/predictions_qwen3-asr-0.6b_ft-noisy-enc32-lm8-cons-lr1e-4.csv (7650 rows)
loaded /kaggle/working/predictions_qwen3-asr-0.6b_zero-shot.csv
WER (%)


snr                                                               clean  20 dB  10 dB   5 dB    0 dB
model          variant                        dataset noise_type                                    
qwen3-asr-0.6b ft-noisy-enc32-lm8-cons-lr1e-4 fleurs  babble        NaN   8.31  11.09  19.31   63.40
                                                      clean        7.90    NaN    NaN    NaN     NaN
                                                      traffic       NaN   8.13   9.51  12.04   18.78
               zero-shot                      fleurs  babble        NaN   8.66  13.28  30.65  109.59
                                                      clean        8.33    NaN    NaN    NaN     NaN
                                                      traffic       NaN   8.81  10.09  12.87   19.73

CER (%)


snr                                                               clean  20 dB  10 dB   5 dB   0 dB
model          variant                        dataset noise_type                                   
qwen3-asr-0.6b ft-noisy-enc32-lm8-cons-lr1e-4 fleurs  babble        NaN   4.84   6.81  12.89  47.51
                                                      clean        4.59    NaN    NaN    NaN    NaN
                                                      traffic       NaN   4.72   5.69   7.54  12.35
               zero-shot                      fleurs  babble        NaN   4.62   8.09  21.40  87.82
                                                      clean        4.39    NaN    NaN    NaN    NaN
                                                      traffic       NaN   4.73   5.54   7.49  12.40

,dataset,noise_type,snr_db,utterances,wer_a,wer_b,diff,ci_low,ci_high,significant
0,fleurs,babble,0.0,850,109.59,63.40,-46.19,-58.69,-34.40,True
1,fleurs,babble,5.0,850,30.65,19.31,-11.34,-16.71,-7.41,True
2,fleurs,babble,10.0,850,13.28,11.09,-2.19,-4.49,-0.79,True
3,fleurs,babble,20.0,850,8.66,8.31,-0.35,-0.73,0.08,False
4,fleurs,clean,NaN,850,8.33,7.90,-0.43,-0.77,-0.06,True
5,fleurs,traffic,0.0,850,19.73,18.78,-0.95,-1.58,-0.31,True
6,fleurs,traffic,5.0,850,12.87,12.04,-0.83,-1.30,-0.32,True
7,fleurs,traffic,10.0,850,10.09,9.51,-0.59,-1.01,-0.15,True
8,fleurs,traffic,20.0,850,8.81,8.13,-0.68,-1.08,-0.27,True


,utt_id,condition,utt_wer,ref,hyp_norm
4283,fleurs_9947910696080043082,babble_0dB,1122.9,chúng ta dựng nhà từ thực vật và may quần áo từ thực vật hầu hết thức ăn của chúng ta là từ thực vật không có thực v...,người hâm mộ hét lên khi họ thấy một cầu thủ xuất sắc hơn cả anh ấy trong một trận đấu có thể là một trận đấu có thể...
7190,fleurs_10865708684535274985,babble_0dB,804.4,tác phẩm của ông có phẩm chất và chi tiết được tán dương đến mức ông là một trong số rất ít tên tuổi lớn nổi bật tro...,tác phẩm của ông là một cuốn sách về chi tiết của một cánh rừng và tin tốt là ông làm một chiếc xe tải và có thể sẽ ...
7388,fleurs_16804968178539575670,babble_0dB,635.1,hiệp hội thể dục hoa kỳ và usoc có chung một mục tiêu đó là làm cho môn thể dục và những môn thể thao khác trở nên c...,từ theo nghĩa là một người đã chết hoặc đã qua đời và được coi là một người đã chết hoặc đã qua đời và được coi là m...
2636,fleurs_13884391343743752501,babble_0dB,240.0,rolando mendoza đã bắn vào các du khách bằng khẩu súng trường m16 của hắn,nó có thể được sử dụng để tạo ra những hình ảnh có độ sắc nét cao hơn và có thể được sử dụng để tạo ra những hình ản...
3554,fleurs_1562013951766668712,babble_0dB,173.7,tuy nhiên về khía cạnh này thì khu vực schengen vận hành gần như một quốc gia duy nhất,tuy nhiên việc đưa cành ma vũ hành kỳ thực hiện được không phải chỉ khiến cho sự hủy hoại của ma vũ hành là tự nhiên...
